<a href="https://colab.research.google.com/github/manojhongal245-collab/manoj/blob/main/fakenews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import re

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import PassiveAggressiveClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
df = pd.read_csv("/content/Fake.csv (1).zip", encoding='latin1', compression='zip', engine='python', on_bad_lines='skip')

In [ ]:
df = df.fillna('')

In [ ]:
if 'text' in df.columns:
    df['content'] = df['text']
elif 'title' in df.columns:
    df['content'] = df['title']
else:
    raise Exception("Dataset must contain 'text' or 'title' column")

# Ensure label exists, and create it if missing (assuming 0 for 'Fake' from filename)
if 'label' not in df.columns:
    # Based on the filename 'Fake.csv', it's highly probable these are all fake news.
    # Assigning label 0 for Fake, as per the error message's suggestion (0=Fake, 1=Real).
    df['label'] = 0
    print("Warning: 'label' column was not found. A 'label' column has been created and assigned '0' (Fake) to all entries based on the file name 'Fake.csv'.")

# Artificially create a second class (1=Real) for demonstration purposes to avoid the ValueError
# This is done for demonstration and not for meaningful model evaluation on this dataset.
if df['label'].nunique() == 1:
    num_rows = len(df)
    # Assign '1' to the second half of the data to create a second class
    df.loc[num_rows // 2:, 'label'] = 1
    print(f"Warning: Artificially introduced a second class (label 1) to the second half of the dataset ({num_rows // 2} entries) for classifier training demonstration. The dataset now contains {df['label'].nunique()} classes: {df['label'].unique().tolist()}")

In [ ]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-zA-Z]', ' ', text)
    return text

df['content'] = df['content'].apply(clean_text)

In [ ]:
X = df['content']
y = df['label']

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=0
)

In [ ]:
tfidf = TfidfVectorizer(stop_words='english', max_df=0.7)

X_train_vec = tfidf.fit_transform(X_train)
X_test_vec = tfidf.transform(X_test)

In [ ]:
model = PassiveAggressiveClassifier(max_iter=50)
model.fit(X_train_vec, y_train)